# PEFT: QLoRA for Text Generation

In this notebook, we show the differences between standard fine-tuning and PEFT using  **QLoRA (Quantized LoRA)** to fine-tune a **causal language model** for a **instruction tuning**.
We combine:
- 4-bit quantization (bitsandbytes)
- LoRA low-rank adapters (PEFT)

QLoRA setup will allow to fine-tune a multi-billion parameter LLM on a **single consumer GPU**.


## 1. Environment Setup

In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

## 2. Load Dataset (Instruction Tuning)

We fine-tune a **causal language model** for **instruction-style generation**, using 
the [tatsu-lab/alpaca dataset](https://huggingface.co/datasets/tatsu-lab/alpaca/viewer/default/train?row=0) containing samples of instructions + responses. 

Each training sample consists of:
- A *prompt / instruction*
- A *target completion*


In [ ]:
dataset = load_dataset('tatsu-lab/alpaca', split='train[:2000]')

## 3. Formatting Prompts

In [ ]:
def format_example(example):
    prompt = 'Instruction: ' + example['instruction'] + ' Input: ' + example['input'] + 'Output: '
    completion = example['output']
    return {'prompt': prompt, 'completion': completion}
dataset = dataset.map(format_example, remove_columns=dataset.column_names)

## 4. Load the Model

We will be using [facebook-opt-1.3b](https://huggingface.co/facebook/opt-1.3b), a 1.3B parameter model

#### 4.1 Load Full Model with no quantization for standard full fine-tuning

In [ ]:
model_name = 'facebook/opt-1.3b'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map='auto'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

#### 4.2 Load Quantized Base Model (4-bit) for QLoRA fine-tuning

we will be using [BitsAndBytes](https://huggingface.co/docs/transformers/en/main_classes/quantization#quantization) to quantize the weights of the original model into 4-bits

In [ ]:
model_name = 'facebook/opt-1.3b'
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map='auto'
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

## 5. Load the Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

## 6. Supervised Fine-Tuning (SFT)
We use **TRL's `SFTTrainer`**, which is optimized for instruction tuning and generation tasks.

#### 6.1 Standard Full SFT (No PEFT)


In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    bf16=True,
    output_dir='./qlora-alpaca',
    save_strategy='epoch',
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

#### 6.2 PEFT SFT with QLoRA


##### LoRA configuration

We need to configure the parameters of LoRA (rank, to which modules LoRA is applied, ...), using the class `LoraConfig`. See https://huggingface.co/docs/peft/package_reference/lora for full documentation

In [ ]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=['q_proj','v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM'
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

##### Fine-tuning

After configuring LoRA, we fine-tune in the usual way

In [ ]:
training_args = TrainingArguments(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=20,
    bf16=True,  
    output_dir='./qlora-alpaca',
    save_strategy='epoch',
    report_to='none'
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer,
)
trainer.train()

## 7. Text Generation Test

In [ ]:
prompt = 'Instruction: ' + 'Explain LoRA in simple terms' + ' Input: ' + 'Output: '

### Response:'
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## 8. Save the final model

In [ ]:
model.save_pretrained('sft_model')

## 9. Exercises
1. Increase model size (e.g. OPT‑2.7B or LLaMA‑style models)
2. Compare ranks r = 4, 8, 16
3. Measure GPU memory with and without quantization
